In [7]:
import requests
from bs4 import BeautifulSoup
import re
import time
from urllib.parse import urlparse
from typing import Optional
from fuzzywuzzy import fuzz, process
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

def is_valid_url(url: str) -> bool:
    try:
        result = urlparse(url)
        return all([result.scheme, result.netloc])
    except ValueError:
        return False

def find_careers_page(company_name: str) -> Optional[str]:
    company_name_lower = company_name.lower().replace(' ', '')
    base_url = f"https://www.{company_name_lower}.com"
    possible_paths = ["/careers", "/jobs", "/about-us/careers", "/join-us", "/careers-jobs", "about/careers#open-positions"]
    possible_subdomains = ["careers.", "jobs."]

    potential_urls = [base_url + path for path in possible_paths]
    potential_urls.extend([
        base_url.replace("www.", subdomain)
        for subdomain in possible_subdomains
        if is_valid_url(base_url.replace("www.", subdomain) + possible_paths[0])
    ])
    potential_urls.append(f"https://{company_name_lower}.com/careers")

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    for url in potential_urls:
        if is_valid_url(url):
            try:
                response = requests.get(url, headers=headers, timeout=5)
                response.raise_for_status()
                if any(k in response.url.lower() or k in response.text.lower() for k in ["careers", "jobs"]):
                    return response.url
            except requests.exceptions.RequestException:
                time.sleep(1)

    return None

def sanitize_title(title: str) -> str:
    title = re.sub(r'[^a-zA-Z0-9\s\-\|]', '', title)
    title = re.sub(r'\s+', ' ', title).strip()
    return title.lower()

def validate_job_posting(job_title: str, company_name: str) -> str:
    careers_url = find_careers_page(company_name)
    if not careers_url:
        return "Could not automatically find the company's careers page. Please check manually."

    options = Options()
    options.headless = True
    driver = webdriver.Chrome(options=options)

    try:
        driver.get(careers_url)
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        job_title_sanitized = sanitize_title(job_title)
        potential_titles = []

        for tag in soup.find_all(['h2', 'h3', 'a', 'p', 'div']):
            text = tag.get_text(separator=' ', strip=True)
            if len(text.split()) <= 20:
                potential_titles.append(sanitize_title(text))

        best_match, best_score = process.extractOne(job_title_sanitized, potential_titles, scorer=fuzz.token_sort_ratio)

        if best_score >= 85:
            return f"✅ Fuzzy match found! '{job_title}' ≈ '{best_match}' ({best_score}% match)\nCareers page: {careers_url}"
        else:
            return f"⚠️ No close matches found for '{job_title}' (best: '{best_match}' at {best_score}% similarity).\nCheck manually: {careers_url}"

    except Exception as e:
        return f"❌ An error occurred during scraping: {e}"

    finally:
        driver.quit()

def run_validation():
    job_title_input = input("Enter the job posting title from LinkedIn: ")
    company_name_input = input("Enter the name of the company: ")
    validation_result = validate_job_posting(job_title_input, company_name_input)
    print("\nValidation Result:")
    print(validation_result)

if __name__ == "__main__":
    run_validation()


Enter the job posting title from LinkedIn:  Associate Data Engineer
Enter the name of the company:  Paylocity



Validation Result:
⚠️ No close matches found for 'Associate Data Engineer' (best: 'become a partner' at 46% similarity).
Check manually: https://www.paylocity.com/careers/


In [7]:
import requests
from bs4 import BeautifulSoup
import re
import time
from urllib.parse import urlparse
from typing import Optional
from fuzzywuzzy import fuzz, process
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

def is_valid_url(url: str) -> bool:
    try:
        result = urlparse(url)
        return all([result.scheme, result.netloc])
    except ValueError:
        return False

def find_careers_page(company_name: str) -> Optional[str]:
    company_name_lower = company_name.lower().replace(' ', '')
    base_url = f"https://www.{company_name_lower}.com"
    possible_paths = ["/careers", "/jobs", "/about-us/careers", "/join-us", "/careers-jobs", "about/careers#open-positions"]
    possible_subdomains = ["careers.", "jobs."]

    potential_urls = [base_url + path for path in possible_paths]
    potential_urls.extend([
        base_url.replace("www.", subdomain)
        for subdomain in possible_subdomains
        if is_valid_url(base_url.replace("www.", subdomain) + possible_paths[0])
    ])
    potential_urls.append(f"https://{company_name_lower}.com/careers")

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    for url in potential_urls:
        if is_valid_url(url):
            try:
                response = requests.get(url, headers=headers, timeout=5)
                response.raise_for_status()
                if any(k in response.url.lower() or k in response.text.lower() for k in ["careers", "jobs"]):
                    return response.url
            except requests.exceptions.RequestException:
                time.sleep(1)

    return None

def sanitize_title(title: str) -> str:
    title = re.sub(r'[^a-zA-Z0-9\s\-\|]', '', title)
    title = re.sub(r'\s+', ' ', title).strip()
    return title.lower()

def validate_job_posting(job_title: str, company_name: str) -> str:
    careers_url = find_careers_page(company_name)
    if not careers_url:
        return "Could not automatically find the company's careers page. Please check manually."

    options = Options()
    options.headless = True
    driver = webdriver.Chrome(options=options)

    try:
        driver.get(careers_url)
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        job_title_sanitized = sanitize_title(job_title)
        potential_titles = []

        for tag in soup.find_all(['h2', 'h3', 'a', 'p', 'div']):
            text = tag.get_text(separator=' ', strip=True)
            if len(text.split()) <= 20:
                potential_titles.append(sanitize_title(text))

        best_match, best_score = process.extractOne(job_title_sanitized, potential_titles, scorer=fuzz.token_sort_ratio)

        if best_score >= 85:
            return f"✅ Fuzzy match found! '{job_title}' ≈ '{best_match}' ({best_score}% match)\nCareers page: {careers_url}"
        else:
            return f"⚠️ No close matches found for '{job_title}' (best: '{best_match}' at {best_score}% similarity).\nCheck manually: {careers_url}"

    except Exception as e:
        return f"❌ An error occurred during scraping: {e}"

    finally:
        driver.quit()

def run_validation():
    job_title_input = input("Enter the job posting title from LinkedIn: ")
    company_name_input = input("Enter the name of the company: ")
    validation_result = validate_job_posting(job_title_input, company_name_input)
    print("\nValidation Result:")
    print(validation_result)

if __name__ == "__main__":
    run_validation()


Enter the job posting title from LinkedIn:  Associate Data Engineer
Enter the name of the company:  Paylocity



Validation Result:
⚠️ No close matches found for 'Associate Data Engineer' (best: 'become a partner' at 46% similarity).
Check manually: https://www.paylocity.com/careers/
